# RAG Pipeline Example

This notebook demonstrates the Retrieval-Augmented Generation (RAG) pattern using a Kale
pipeline. It builds a document index from a text corpus, retrieves relevant documents for
a query, and generates a context-aware response.

This example uses scikit-learn for text vectorization and similarity search to illustrate
the RAG pattern without requiring external GenAI libraries. In a production RAG pipeline,
you would replace the vectorizer with an embedding model (e.g., sentence-transformers) and
the response template with an LLM call (e.g., via langchain or vLLM).

In [ ]:
import textwrap

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Pipeline Parameters

These parameters control the RAG pipeline behavior. Kale converts top-level variable
assignments into pipeline parameters, so the pipeline can be re-run with different queries.

In [ ]:
QUERY = "How do I convert a notebook to a pipeline?"
TOP_K = 3

### Document Corpus

In a production RAG pipeline, documents would be loaded from a database, file system, or
API. This example uses an inline corpus of Kale documentation snippets.

In [ ]:
DOCUMENTS = [
    {
        "id": "getting-started",
        "title": "Getting Started with Kale",
        "content": "Kale converts Jupyter Notebooks into Kubeflow Pipelines. Install Kale, "
        "open a notebook, and use cell tags to annotate which cells belong to which pipeline "
        "step. Kale handles the rest: dependency detection, containerization, and deployment.",
    },
    {
        "id": "cell-tags",
        "title": "Using Cell Tags",
        "content": "Kale uses notebook cell tags to define pipeline steps. Add a tag like "
        "'step:data-loading' to a cell to assign it to the data-loading step. Cells without "
        "tags are included in the pipeline's initialization. Multiple cells can share the "
        "same step tag.",
    },
    {
        "id": "pipeline-parameters",
        "title": "Pipeline Parameters",
        "content": "Top-level variable assignments in a notebook become pipeline parameters "
        "in Kale. This means you can re-run the pipeline with different values without "
        "editing the notebook. Parameters are detected automatically by Kale's code analysis.",
    },
    {
        "id": "dependencies",
        "title": "Dependency Management",
        "content": "Kale automatically detects which pipeline steps depend on each other by "
        "analyzing variable usage across cells. If step B reads a variable that step A "
        "writes, Kale creates a dependency edge from A to B in the pipeline DAG.",
    },
    {
        "id": "compilation",
        "title": "Compiling Notebooks",
        "content": "Use the Kale CLI to compile a notebook: 'kale --nb my_notebook.ipynb'. "
        "This produces a KFP pipeline YAML that can be uploaded to Kubeflow Pipelines. "
        "Add '--run_pipeline' to compile and submit in one step.",
    },
    {
        "id": "serving",
        "title": "Model Serving with Kale",
        "content": "Kale supports model serving through KServe integration. After training "
        "a model in a pipeline step, you can deploy it as an inference service by adding "
        "serving annotations to the notebook.",
    },
    {
        "id": "volumes",
        "title": "Volume Mounts",
        "content": "Kale pipelines can mount persistent volumes for data sharing between "
        "steps. Configure volume mounts in the Kale sidebar to specify which volumes are "
        "available to each pipeline step.",
    },
    {
        "id": "debugging",
        "title": "Debugging Pipelines",
        "content": "When a pipeline step fails, Kale preserves the step's output logs in "
        "Kubeflow Pipelines. You can also run individual steps locally by executing the "
        "notebook cells that belong to that step.",
    },
]

### Step 1: Build Document Index

This step creates a TF-IDF vector index from the document corpus. In a production RAG
pipeline, you would use an embedding model (e.g., `sentence-transformers`) and a vector
database (e.g., ChromaDB, FAISS) instead.

In [ ]:
corpus_texts = [doc["content"] for doc in DOCUMENTS]
corpus_titles = [doc["title"] for doc in DOCUMENTS]

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(corpus_texts)
print(f"Indexed {len(DOCUMENTS)} documents with {tfidf_matrix.shape[1]} features")

### Step 2: Retrieve Relevant Documents

Given a query, this step finds the most similar documents using cosine similarity on
the TF-IDF vectors. In a production system, this would be a vector database similarity
search using dense embeddings.

In [ ]:
query_vector = vectorizer.transform([QUERY])
similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
top_indices = np.argsort(similarities)[::-1][:TOP_K]

retrieved_docs = []
print(f"Query: {QUERY}\n")
print(f"Top {TOP_K} results:")
for rank, idx in enumerate(top_indices, 1):
    doc = DOCUMENTS[idx]
    score = similarities[idx]
    retrieved_docs.append(doc)
    print(f"  {rank}. [{score:.3f}] {doc['title']}")

### Step 3: Generate Response

This step constructs a response using the retrieved documents as context. In a production
RAG pipeline, you would send the query and retrieved context to an LLM (e.g., via
langchain, vLLM, or a direct API call). This example uses a template-based approach to
demonstrate the pattern.

In [ ]:
context = "\n\n".join(
    f"[{doc['title']}]: {doc['content']}" for doc in retrieved_docs
)

response = textwrap.dedent(f"""\    Based on the retrieved documentation:

    {context}

    Answer: To convert a notebook to a pipeline with Kale, use cell tags to annotate
    which cells belong to which pipeline step, then compile with 'kale --nb notebook.ipynb'.
    Kale automatically detects dependencies between steps and creates the pipeline DAG.
""")

print("Generated Response:")
print(response)

### Summary

This notebook demonstrated the three core steps of a RAG pipeline:
1. **Indexing** -- Build a searchable representation of your document corpus
2. **Retrieval** -- Find the most relevant documents for a given query
3. **Generation** -- Use the retrieved context to produce an informed response

To build a production RAG pipeline with Kale, replace:
- `TfidfVectorizer` with an embedding model (e.g., `sentence-transformers`)
- In-memory corpus with a vector database (e.g., ChromaDB, FAISS)
- Template-based generation with an LLM call (e.g., via `langchain` or `vLLM`)

Each of these steps would become a separate Kale pipeline step using cell tags.

In [ ]:
print(f"RAG pipeline complete. Retrieved {len(retrieved_docs)} documents for query.")
print(f"Response length: {len(response)} characters")